In [1]:
!pip install google-api-python-client textblob pandas

In [8]:
from googleapiclient.discovery import build
import pandas as pd
from textblob import TextBlob
import re

# ==========================================
# PHASE 1: OFFICIAL API EXTRACTION
# ==========================================
API_KEY = 'API key here'
youtube = build('youtube', 'v3', developerKey=API_KEY)

# Enter the Video ID of a Pakistani Tech Review or E-commerce Ad
# (e.g., for "https://www.youtube.com/watch?v=dQw4w9WgXcQ", the ID is "dQw4w9WgXcQ")
VIDEO_ID = 'X_G0TnMMN1w'
def get_video_comments(video_id):
    comments_list = []

    # Call the YouTube API to fetch comment threads
    request = youtube.commentThreads().list(
        part="snippet",
        videoId=video_id,
        maxResults=100, # Pulls 100 comments per request
        textFormat="plainText"
    )

    response = request.execute()

    # Loop through the API JSON response to extract the exact data we need
    while request is not None:
        response = request.execute()
        for item in response['items']:
            comment = item['snippet']['topLevelComment']['snippet']
            comments_list.append({
                'Author': comment['authorDisplayName'],
                'Comment_Text': comment['textDisplay'],
                'Like_Count': comment['likeCount'],
                'Published_At': comment['publishedAt']
            })

        # Handle pagination to get more than 100 comments if needed
        if 'nextPageToken' in response:
            request = youtube.commentThreads().list(
                part="snippet",
                videoId=video_id,
                pageToken=response['nextPageToken'],
                maxResults=100,
                textFormat="plainText"
            )
        else:
            break

    return pd.DataFrame(comments_list)

print("Connecting to YouTube API...")
df_raw = get_video_comments(VIDEO_ID)
print(f"Extraction Complete: {len(df_raw)} comments fetched.")

# ==========================================
# PHASE 2: DATA CLEANING
# ==========================================
# Remove empty comments and drop missing values
df_clean = df_raw.dropna(subset=['Comment_Text']).copy()

# Remove URLs, special characters, and extra spaces
def clean_text(text):
    text = re.sub(r'http\S+', '', str(text))
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    return text.strip()

df_clean['Cleaned_Comment'] = df_clean['Comment_Text'].apply(clean_text)

# ==========================================
# PHASE 3: SENTIMENT ANALYSIS (NLP)
# ==========================================
def analyze_sentiment(text):
    # Calculate polarity: -1.0 (Highly Negative) to 1.0 (Highly Positive)
    polarity = TextBlob(text).sentiment.polarity
    if polarity > 0.1:
        return 'Positive (Purchase Intent / Excitement)'
    elif polarity < -0.1:
        return 'Negative (Complaint / Dislike)'
    else:
        return 'Neutral (Informational / Inquiry)'

df_clean['Sentiment_Label'] = df_clean['Cleaned_Comment'].apply(analyze_sentiment)

# ==========================================
# PHASE 4: BUSINESS AGGREGATION & EXPORT
# ==========================================
print("\n=== AUDIENCE SENTIMENT BREAKDOWN ===")
sentiment_counts = df_clean['Sentiment_Label'].value_counts()
print(sentiment_counts)

# Identify the most "liked" negative and positive comments
top_positive = df_clean[df_clean['Sentiment_Label'].str.contains('Positive')].sort_values(by='Like_Count', ascending=False).head(1)
top_negative = df_clean[df_clean['Sentiment_Label'].str.contains('Negative')].sort_values(by='Like_Count', ascending=False).head(1)

print("\n=== MOST UPVOTED POSITIVE COMMENT ===")
if not top_positive.empty: print(top_positive['Comment_Text'].values[0])

print("\n=== MOST UPVOTED NEGATIVE COMMENT (Major Pain Point) ===")
if not top_negative.empty: print(top_negative['Comment_Text'].values[0])

# Export for GitHub
file_name = "youtube_ad_sentiment_analysis.csv"
df_clean.to_csv(file_name, index=False)
print(f"\nPipeline Complete! Data saved as {file_name}.")

Connecting to YouTube API...
Extraction Complete: 751 comments fetched.

=== AUDIENCE SENTIMENT BREAKDOWN ===
Sentiment_Label
Neutral (Informational / Inquiry)          521
Positive (Purchase Intent / Excitement)    168
Negative (Complaint / Dislike)              62
Name: count, dtype: int64

=== MOST UPVOTED POSITIVE COMMENT ===
Not an Apple or Samsung fan but why does it sound that Bilal ko Samsung ne lift nhi krwae aur woh bas khunas nikal rha h jese isko majburi m paise khrch krwaye hei kisi ne😂

=== MOST UPVOTED NEGATIVE COMMENT (Major Pain Point) ===
I m not samsung user but this is  unfair and one sided vlog. Fazool video

Pipeline Complete! Data saved as youtube_ad_sentiment_analysis.csv.
